In [1]:
import pandas as pd

### **Data Load**

In [2]:
df = pd.read_csv("data/Eng-Fra.csv")
df.head()

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [3]:
print(df.columns)
print(df.shape)

Index(['English', 'French'], dtype='object')
(239189, 2)


### **Preprocessing**

In [4]:
from preprocessing.normalize import normalize_text
df['English'] = df['English'].apply(normalize_text)
df['French'] = df['French'].apply(normalize_text)

In [5]:
df['English'] = df['English'].astype(str)
df['French'] = df['French'].astype(str)

In [6]:
df['English'] = df['English'].str.strip()
df['French'] = df['French'].str.strip()

In [7]:
df.to_csv("data/clean_eng_to_french.csv")

### **Creating `train.txt`**

In [8]:
with open('data/train.txt','w', encoding = 'utf-8') as f:
    for eng, fra in zip(df['English'],df['French']):
        f.write(eng + "\n")
        f.write(fra + "\n")

### **Training SentencePiece**

In [9]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input = 'data/train.txt',
    model_prefix = 'spm',
    vocab_size = 8000,
    model_type = 'bpe',
    pad_id = 0,
    unk_id = 1,
    bos_id = 2,
    eos_id = 3
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: data/train.txt
  input_format: 
  model_prefix: spm
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_

### **Creating tokens**

In [10]:
from tokenizer import Tokenizer

tokenizer = Tokenizer("spm.model")
tokenizer.encode("I love cats and dogs")


[7959, 1, 745, 4016, 308, 3053]

In [11]:
eng_sentence = []
fr_sentence = []
for eng,fr in zip(df['English'], df['French']):
    eng_sentence.append(eng)
    fr_sentence.append(fr)

In [12]:
from dataset import TranslationDataset

translation_dataset = TranslationDataset(english_sentence = eng_sentence, french_sentence = fr_sentence, tokenizer = tokenizer, max_len = 50)

In [13]:
len(translation_dataset)

239189

In [14]:
sample = translation_dataset[0]
sample

{'encoder_input': tensor([143,   3]),
 'decoder_input': tensor([  2, 614]),
 'target': tensor([614,   3])}

In [15]:
print(sample["encoder_input"].shape)
print(sample["decoder_input"].shape)
print(sample["target"].shape)

torch.Size([2])
torch.Size([2])
torch.Size([2])


In [16]:
from torch.utils.data import DataLoader

loader = DataLoader(
    translation_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=translation_dataset.collate_fn
)

batch = next(iter(loader))

print(batch["encoder_inputs"].shape)
print(batch["decoder_inputs"].shape)
print(batch["targets"].shape)

torch.Size([4, 8])
torch.Size([4, 8])
torch.Size([4, 8])


In [17]:
from model.embedding import InputEmbedding
from model.positional_encoding import PositionalEncoding

embedding = InputEmbedding(vocab_size=tokenizer.vocab_size, d_model=512)
positional_encoding = PositionalEncoding(d_model=512, max_len=50)

x = embedding(batch["encoder_inputs"])
x = positional_encoding(x)

print(x.shape)

torch.Size([4, 8, 512])
